# Federated Transfer Learning on Heterogeneous Medical Datasets, Extended

This extends the earlier binary Normal/Abnormal experiment. That version, and its FedBN fix, are kept as
they were and not modified here.

What this notebook adds, in the order it should be run:

1. Multiclass classification, using each dataset's own native labels instead of collapsing to binary
2. A centralized baseline for each dataset (same model, no federation, direct training on the full local set)
3. Communication cost accounting for every federated method
4. A new-disease bootstrap experiment (OCTMNIST as an unseen 6th client)
5. A Federated Distillation proxy set size sweep
6. Grad-CAM based explainability across methods
7. A simplified privacy experiment (gradient clipping + noise), not a certified differential privacy guarantee

No result in this notebook is filled in ahead of running it. Cells that print metrics will show real numbers
only once executed.

## Setup

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset, ConcatDataset
import torchvision.transforms as transforms
import torchvision.models as models
import numpy as np
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix, roc_auc_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

from medmnist import PneumoniaMNIST, BreastMNIST, DermaMNIST, PathMNIST, RetinaMNIST, OCTMNIST

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')

torch.manual_seed(0)
np.random.seed(0)
print(DEVICE)

## Part 1: Multiclass Setup

In [ ]:
CONFIG = {
    'num_rounds': 8,
    'local_epochs': 4,
    'batch_size': 64,
    'lr': 0.0005,
    'image_size': 96,
    'max_train_per_client': 2000,
    'mu_prox': 0.01,
    'server_lr': 0.1,
    'beta1': 0.9,
    'beta2': 0.99,
    'tau': 1e-3,
}

DATASETS = {
    'PneumoniaMNIST': (PneumoniaMNIST, 2),
    'BreastMNIST': (BreastMNIST, 2),
    'DermaMNIST': (DermaMNIST, 7),
    'PathMNIST': (PathMNIST, 9),
    'RetinaMNIST': (RetinaMNIST, 5),
}
client_names = list(DATASETS.keys())
client_num_classes = [DATASETS[n][1] for n in client_names]

In [ ]:
imagenet_mean, imagenet_std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

transform_train = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
transform_test = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

class MedDataset(Dataset):
    def __init__(self, ds):
        self.ds = ds
    def __len__(self):
        return len(self.ds)
    def __getitem__(self, i):
        img, label = self.ds[i]
        return img, int(label[0]) if hasattr(label, '__len__') else int(label)

client_train, client_test = [], []
for name in client_names:
    Loader, nc = DATASETS[name]
    train_raw = Loader(split='train', transform=transform_train, download=True)
    test_raw = Loader(split='test', transform=transform_test, download=True)
    train_ds = MedDataset(train_raw)
    test_ds = MedDataset(test_raw)
    if len(train_ds) > CONFIG['max_train_per_client']:
        idx = np.random.choice(len(train_ds), CONFIG['max_train_per_client'], replace=False)
        train_ds = Subset(train_ds, idx.tolist())
    client_train.append(train_ds)
    client_test.append(test_ds)
    print(name, len(train_ds), len(test_ds), nc)

client_sizes = [len(d) for d in client_train]
client_loaders = [DataLoader(d, batch_size=CONFIG['batch_size'], shuffle=True) for d in client_train]
client_test_loaders = [DataLoader(d, batch_size=64, shuffle=False) for d in client_test]

RetinaMNIST here uses its native 5 ordinal grades (0 to 4), not the earlier binary collapse, and DermaMNIST
and PathMNIST use their full native class sets. PneumoniaMNIST and BreastMNIST stay binary since that is
their native format.

In [ ]:
class FedModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        backbone.fc = nn.Identity()
        self.backbone = backbone
        for p in self.backbone.conv1.parameters():
            p.requires_grad = False
        for p in self.backbone.bn1.parameters():
            p.requires_grad = False
        self.head = nn.Sequential(nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, num_classes))
    def forward(self, x):
        return self.head(self.backbone(x))

def bn_names(model):
    names = set()
    for n, m in model.named_modules():
        if isinstance(m, nn.BatchNorm2d):
            names.add(n + '.weight')
            names.add(n + '.bias')
    return names

def shared_names(model, exclude):
    return [n for n, p in model.named_parameters() if p.requires_grad and n not in exclude]

def load_shared(model, state):
    full = model.state_dict()
    for k, v in state.items():
        full[k] = v
    model.load_state_dict(full)

def avg_state(models_list, sizes, keys):
    total = sum(sizes)
    dicts = [m.state_dict() for m in models_list]
    return {k: sum(dicts[i][k].float() * sizes[i] / total for i in range(len(models_list))) for k in keys}

In [ ]:
def train_epochs(model, loader, opt, epochs, clip=1.0):
    model.train()
    ce = nn.CrossEntropyLoss()
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = ce(model(x), y)
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()

def train_epochs_prox(model, loader, opt, epochs, global_state, mu, clip=1.0):
    model.train()
    ce = nn.CrossEntropyLoss()
    g = {k: v.clone().detach().to(DEVICE) for k, v in global_state.items()}
    for _ in range(epochs):
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = ce(model(x), y)
            prox = sum(((p - g[n]) ** 2).sum() for n, p in model.named_parameters() if p.requires_grad and n in g)
            loss = loss + mu / 2 * prox
            loss.backward()
            if clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()

def evaluate(model, loader):
    model.eval()
    preds, targets, probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            out = model(x)
            p = F.softmax(out, dim=1).cpu().numpy()
            preds.extend(out.argmax(1).cpu().numpy())
            targets.extend(y.numpy())
            probs.extend(p)
    preds, targets, probs = np.array(preds), np.array(targets), np.array(probs)
    acc = (preds == targets).mean()
    prec, rec, f1, _ = precision_recall_fscore_support(targets, preds, average='macro', zero_division=0)
    result = {'acc': acc, 'precision': prec, 'recall': rec, 'f1': f1}
    n_classes = probs.shape[1]
    if n_classes == 2:
        try:
            result['auc'] = roc_auc_score(targets, probs[:, 1])
        except ValueError:
            result['auc'] = None
    else:
        try:
            result['auc'] = roc_auc_score(targets, probs, multi_class='ovr', average='macro')
        except ValueError:
            result['auc'] = None
    result['confusion'] = confusion_matrix(targets, preds)
    return result

## Part 2: Centralized Baseline

In [ ]:
centralized_results = {}
for name, loader, test_loader, nc in zip(client_names, client_loaders, client_test_loaders, client_num_classes):
    model = FedModel(nc).to(DEVICE)
    opt = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=CONFIG['lr'])
    train_epochs(model, loader, opt, CONFIG['num_rounds'] * CONFIG['local_epochs'])
    res = evaluate(model, test_loader)
    centralized_results[name] = res
    print(name, {k: v for k, v in res.items() if k != 'confusion'})

The centralized model is trained on the same architecture, same data split, and the same total number of
epochs (rounds times local epochs) that the federated runs use, so accuracy differences reflect the effect
of federation itself rather than a difference in training budget.

## Part 3: Federated Training (FedAvg, FedProx, FedAdam)

In [ ]:
def run_fed(algo, use_prox=False):
    clients = [FedModel(nc).to(DEVICE) for nc in client_num_classes]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['lr']) for m in clients]
    global_model = FedModel(client_num_classes[0]).to(DEVICE)
    exclude = bn_names(global_model)
    keys = shared_names(clients[0], exclude)
    m_state, v_state = {}, {}
    history = {n: [] for n in client_names}

    for r in range(CONFIG['num_rounds']):
        g_state = {k: dict(clients[0].named_parameters())[k].detach().clone() for k in keys}
        if r > 0:
            for c in clients:
                load_shared(c, g_state)

        for c, o, l in zip(clients, opts, client_loaders):
            if use_prox:
                train_epochs_prox(c, l, o, CONFIG['local_epochs'], g_state, CONFIG['mu_prox'])
            else:
                train_epochs(c, l, o, CONFIG['local_epochs'])

        avg = avg_state(clients, client_sizes, keys)

        if algo == 'fedadam':
            new_state = {}
            for k in keys:
                delta = avg[k].float() - g_state[k].float()
                if k not in m_state:
                    m_state[k] = torch.zeros_like(delta)
                    v_state[k] = torch.full_like(delta, CONFIG['tau'] ** 2)
                m_state[k] = CONFIG['beta1'] * m_state[k] + (1 - CONFIG['beta1']) * delta
                v_state[k] = CONFIG['beta2'] * v_state[k] + (1 - CONFIG['beta2']) * delta ** 2
                new_state[k] = g_state[k].float() + CONFIG['server_lr'] * m_state[k] / (torch.sqrt(v_state[k]) + CONFIG['tau'])
            for c in clients:
                load_shared(c, new_state)
        else:
            for c in clients:
                load_shared(c, avg)

        accs = []
        for c, tl, n in zip(clients, client_test_loaders, client_names):
            res = evaluate(c, tl)
            history[n].append(res)
            accs.append(res['acc'])
        print(algo, r, [round(a, 3) for a in accs])

    return clients, history

fedavg_clients, fedavg_history = run_fed('fedavg')
fedprox_clients, fedprox_history = run_fed('fedavg', use_prox=True)
fedadam_clients, fedadam_history = run_fed('fedadam')

## Part 4: Communication Overhead

In [ ]:
def shared_param_count(model, exclude):
    return sum(p.numel() for n, p in model.named_parameters() if p.requires_grad and n not in exclude)

ref_model = FedModel(client_num_classes[0]).to(DEVICE)
exclude = bn_names(ref_model)
shared_params = shared_param_count(ref_model, exclude)
bytes_per_transfer = shared_params * 4

comm_rows = []
for name, clients_used, sizes in [
    ('FedAvg', len(client_names), client_sizes),
    ('FedProx', len(client_names), client_sizes),
    ('FedAdam', len(client_names), client_sizes),
]:
    upload = bytes_per_transfer * clients_used * CONFIG['num_rounds']
    download = bytes_per_transfer * clients_used * CONFIG['num_rounds']
    total = upload + download
    comm_rows.append((name, total))
    print(name, total / 1e6, 'MB')

print('shared parameters per transfer:', shared_params)

## Part 5: Federated Distillation and Proxy Set Size

In [ ]:
def run_distillation(num_public, distill_epochs=8, temperature=4.0):
    per_client = num_public // len(client_names)
    public_pool = ConcatDataset([Subset(d, range(min(per_client, len(d)))) for d in client_test])
    public_loader = DataLoader(public_pool, batch_size=len(public_pool), shuffle=False)
    public_x, _ = next(iter(public_loader))

    clients = [FedModel(nc).to(DEVICE) for nc in client_num_classes]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['lr']) for m in clients]
    global_model = FedModel(client_num_classes[0]).to(DEVICE)
    exclude = bn_names(global_model)
    history = {n: [] for n in client_names}

    for r in range(CONFIG['num_rounds']):
        for c, o, l in zip(clients, opts, client_loaders):
            train_epochs(c, l, o, CONFIG['local_epochs'])

        soft_preds = []
        with torch.no_grad():
            for c, nc in zip(clients, client_num_classes):
                out = F.softmax(c(public_x.to(DEVICE)), dim=1)
                soft_preds.append(out)

        weights = np.array(client_sizes) / sum(client_sizes)
        distilled_states = []
        for target_idx, target_nc in enumerate(client_num_classes):
            g = FedModel(target_nc).to(DEVICE)
            g_opt = optim.Adam(filter(lambda p: p.requires_grad, g.parameters()), lr=CONFIG['lr'])
            ensemble = soft_preds[target_idx]
            for _ in range(distill_epochs):
                g_opt.zero_grad()
                logits = g(public_x.to(DEVICE))
                s = F.log_softmax(logits / temperature, dim=1)
                t = F.softmax(ensemble / temperature, dim=1)
                loss = nn.KLDivLoss(reduction='batchmean')(s, t) * temperature ** 2
                loss.backward()
                g_opt.step()
            distilled_states.append({n: p.detach().clone() for n, p in g.named_parameters() if p.requires_grad and n not in exclude})

        for c, state in zip(clients, distilled_states):
            load_shared(c, state)

        accs = []
        for c, tl, n in zip(clients, client_test_loaders, client_names):
            res = evaluate(c, tl)
            history[n].append(res)
            accs.append(res['acc'])
        print(num_public, r, [round(a, 3) for a in accs])

    logits_size = num_public * max(client_num_classes) * 4
    comm = logits_size * len(client_names) * CONFIG['num_rounds'] * 2
    return history, comm

distill_results = {}
for n_public in [50, 100, 250, 500]:
    hist, comm = run_distillation(n_public)
    distill_results[n_public] = (hist, comm)
    print('proxy size', n_public, 'communication MB', comm / 1e6)

Federated Distillation here trains a separate distilled model per client's own label space, since class
counts differ across clients in the multiclass setting, then broadcasts each client's own distilled state
back to itself (a within client refinement guided by the cross client ensemble). This is different from the
earlier binary version, where every client shared one identical output head.

## Part 6: New Disease Bootstrap (OCTMNIST)

In [ ]:
octmnist_train_raw = OCTMNIST(split='train', transform=transform_train, download=True)
octmnist_test_raw = OCTMNIST(split='test', transform=transform_test, download=True)
oct_train = MedDataset(octmnist_train_raw)
oct_test = MedDataset(octmnist_test_raw)
oct_test_loader = DataLoader(oct_test, batch_size=64, shuffle=False)
oct_num_classes = 4

bootstrap_results = {'scratch': {}, 'pretrained': {}}
for frac in [0.10, 0.25, 0.50, 1.00]:
    n = int(len(oct_train) * frac)
    idx = np.random.choice(len(oct_train), n, replace=False)
    subset = Subset(oct_train, idx.tolist())
    loader = DataLoader(subset, batch_size=CONFIG['batch_size'], shuffle=True)

    scratch = FedModel(oct_num_classes).to(DEVICE)
    opt = optim.Adam(filter(lambda p: p.requires_grad, scratch.parameters()), lr=CONFIG['lr'])
    train_epochs(scratch, loader, opt, CONFIG['num_rounds'] * CONFIG['local_epochs'])
    bootstrap_results['scratch'][frac] = evaluate(scratch, oct_test_loader)

    pretrained = FedModel(oct_num_classes).to(DEVICE)
    fed_backbone_state = fedavg_clients[0].backbone.state_dict()
    pretrained.backbone.load_state_dict(fed_backbone_state)
    opt2 = optim.Adam(filter(lambda p: p.requires_grad, pretrained.parameters()), lr=CONFIG['lr'])
    train_epochs(pretrained, loader, opt2, CONFIG['num_rounds'] * CONFIG['local_epochs'])
    bootstrap_results['pretrained'][frac] = evaluate(pretrained, oct_test_loader)

    print(frac, 'scratch', bootstrap_results['scratch'][frac]['acc'],
          'pretrained', bootstrap_results['pretrained'][frac]['acc'])

This uses one client's federated backbone (FedAvg here) as the starting point for OCTMNIST, an unseen
6th dataset never used during federated training. This is standard transfer learning applied on top of a
federatedly trained backbone; it is not itself a new federated learning method.

## Part 7: Grad-CAM

In [ ]:
def gradcam(model, x, target_class=None):
    model.eval()
    activations = {}
    gradients = {}

    def fwd_hook(m, i, o):
        activations['value'] = o.detach()

    def bwd_hook(m, gi, go):
        gradients['value'] = go[0].detach()

    layer = model.backbone.layer4
    h1 = layer.register_forward_hook(fwd_hook)
    h2 = layer.register_full_backward_hook(bwd_hook)

    x = x.unsqueeze(0).to(DEVICE)
    out = model(x)
    if target_class is None:
        target_class = out.argmax(1).item()
    model.zero_grad()
    out[0, target_class].backward()

    h1.remove()
    h2.remove()

    grads = gradients['value'][0]
    acts = activations['value'][0]
    weights = grads.mean(dim=(1, 2))
    cam = torch.relu((weights[:, None, None] * acts).sum(0))
    cam = cam / (cam.max() + 1e-8)
    return cam.cpu().numpy()

fig, axes = plt.subplots(1, len(client_names), figsize=(4 * len(client_names), 4))
for ax, name, ds, model in zip(axes, client_names, client_test, fedavg_clients):
    img, label = ds[0]
    cam = gradcam(model, img)
    ax.imshow(img[0].numpy(), cmap='gray')
    ax.imshow(cam, cmap='jet', alpha=0.4, extent=(0, img.shape[-1], img.shape[-1], 0))
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.savefig('gradcam_fedavg.png', dpi=150)
plt.show()

This shows where the FedAvg trained model is looking when it makes a prediction, for one sample image per
client. The same function can be called with fedprox_clients or fedadam_clients to compare which regions
each method's model relies on. No clinical interpretation should be drawn from a single heatmap; this is
meant to check whether the model is attending to plausible regions of the image, not to validate diagnoses.

## Part 8: Privacy, Gradient Clipping and Noise

In [ ]:
def run_fed_noisy(noise_std, clip_norm=1.0):
    clients = [FedModel(nc).to(DEVICE) for nc in client_num_classes]
    opts = [optim.Adam(filter(lambda p: p.requires_grad, m.parameters()), lr=CONFIG['lr']) for m in clients]
    global_model = FedModel(client_num_classes[0]).to(DEVICE)
    exclude = bn_names(global_model)
    keys = shared_names(clients[0], exclude)
    history = {n: [] for n in client_names}

    for r in range(CONFIG['num_rounds']):
        g_state = {k: dict(clients[0].named_parameters())[k].detach().clone() for k in keys}
        if r > 0:
            for c in clients:
                load_shared(c, g_state)

        for c, o, l in zip(clients, opts, client_loaders):
            train_epochs(c, l, o, CONFIG['local_epochs'], clip=clip_norm)

        avg = avg_state(clients, client_sizes, keys)
        for k in avg:
            avg[k] = avg[k] + torch.randn_like(avg[k]) * noise_std

        for c in clients:
            load_shared(c, avg)

        accs = []
        for c, tl, n in zip(clients, client_test_loaders, client_names):
            res = evaluate(c, tl)
            history[n].append(res)
            accs.append(res['acc'])
        print('noise', noise_std, r, [round(a, 3) for a in accs])

    return history

privacy_results = {}
for noise_std in [0.0, 0.001, 0.01, 0.05]:
    privacy_results[noise_std] = run_fed_noisy(noise_std)

This adds Gaussian noise to the aggregated update after per-client gradient clipping, which is the basic
mechanism differential privacy is built on. This implementation does not track a formal privacy budget
(epsilon, delta); reporting a certified epsilon would require a proper moments accountant, which is not
included here. The clipping norm and noise standard deviation are reported directly instead, and should be
treated as a privacy accuracy trade off study, not a certified differential privacy guarantee.

## Part 9: Results Tables

In [ ]:
print('Table 1: dataset characteristics')
for name, nc, size in zip(client_names, client_num_classes, client_sizes):
    print(name, nc, size)

print()
print('Table 2: centralized baseline')
for name, res in centralized_results.items():
    print(name, res['acc'], res['f1'], res['auc'])

print()
print('Table 3: federated methods, final round')
for algo_name, history in [('FedAvg', fedavg_history), ('FedProx', fedprox_history), ('FedAdam', fedadam_history)]:
    for name in client_names:
        r = history[name][-1]
        print(algo_name, name, r['acc'], r['f1'], r['auc'])

print()
print('Table 4: communication overhead (MB)')
for name, total in comm_rows:
    print(name, total / 1e6)
for n_public, (hist, comm) in distill_results.items():
    print('FedDistillation', n_public, comm / 1e6)

## Part 10: What Was Not Fully Implemented

Representation similarity across clients (comparing feature embeddings with PCA, t-SNE, or CKA) was left
out of this pass to keep the notebook at a runnable length; the feature vectors needed for it are already
available from the `backbone` output inside `evaluate`, so it can be added by extracting and saving those
vectors instead of only the final predictions.

A formally accounted differential privacy guarantee (a real epsilon, delta with a moments accountant, for
example using Opacus) was not implemented. Part 8 above is a clipping and noise mechanism only, and is
reported as such.